# Lesson 5 — Structure: instructions and criteria can be JSON, not just strings

Lessons 2–4 covered the three primitives as plain-string questions. This lesson covers a capability that applies to all three at once: `instructions` and `criteria` don't have to be strings — they can be structured JSON objects and arrays.

Ask your agent if anything's unclear.

<details>
  <summary><strong>Primary source (trimmed):</strong> docs.typesafe.ai/primitives/advanced (click to expand)</summary>

# Advanced: structure

> Instructions, Choice options, Score levels, and Noul criteria all accept JSON structure.

System One models are trained to understand structure.

## Where structure is allowed

| Field | Applies to | Accepted shape |
| --- | --- | --- |
| `instructions` | Choice, Score, Noul | `string`, `object`, `array`, or `null` |
| `criteria` values (option descriptions) | Choice | `string`, `object`, `array`, or `null` |
| `criteria` entries (level descriptions) | Score | `string`, `object`, `array`, or `null` |
| `criteria.true` and `criteria.false` | Noul | `string`, `object`, `array`, or `null` |

## When to structure a question

- **When it helps with clarity.** When a question has multiple parts, putting them in the form of JSON helps with clarity because the keys are labeled.
- **When the question needs supporting data.** A schema, a taxonomy, or a database row is already JSON. Use the JSON entirely or pass in the relevant subfields instead of serializing them into a string template.

## Structured Choice options

A Choice option description can be a structured object. The example below tells the model what each option does and does *not* cover — sharpening the boundary between similar options:

```json
{
  "state": "I ordered the standing desk two weeks ago and tracking still says label created. Was I even charged?",
  "questions": {
    "department": {
      "type": "choice",
      "instructions": {
        "question": "Which team should handle this message?",
        "focus": "Classify the customer's primary request, not every topic mentioned."
      },
      "criteria": {
        "billing": {
          "what": "Charges, invoices, refunds, or subscriptions",
          "not_for": "Order tracking or account access",
          "examples": ["I was charged twice", "Where is my refund?"]
        },
        "orders": {
          "what": "Order status, delivery, cancellation, or returns",
          "not_for": "Charges or account access",
          "examples": ["Where is my package?", "Cancel my order"]
        },
        "account": {
          "what": "Login, password, profile, or security",
          "not_for": "Charges or delivery",
          "examples": ["I can't log in", "Change my email"]
        }
      }
    }
  }
}
```

The field names `what`, `not_for`, `focus`, and `examples` are not part of the API, and none are reserved — you choose them, the same way you choose option names. The model sees the names along with the values, so use short names that label what follows.

## Structured Score levels

Each entry in a Score `criteria` array can be an object too, with fields like `summary` and `signals` instead of a plain description string — useful when a level needs more than one sentence to pin down, such as judging how focused a pull request description is on a single change.

## Structured Noul criteria

Noul's optional `criteria.true` / `criteria.false` can be structured the same way: a `what` field plus `examples`, for when the yes/no boundary is subtle enough that a plain sentence isn't precise enough on its own — for example, distinguishing "asks to reset a password" from "asks you to send your password."

> This page also covers walking a deep taxonomy with chained, structured Choice questions (one question per tree level, each option's value being its own subtree) — genuinely its own topic, covered by the [Hierarchical Classification cookbook](https://docs.typesafe.ai/cookbooks/hierarchical_classification) rather than here. Read the full page at [docs.typesafe.ai/primitives/advanced](https://docs.typesafe.ai/primitives/advanced) any time.

</details>

## The core idea

Nothing here is a fourth primitive — it's a capability all three already had. Every `instructions` and every `criteria` entry you've written so far has been a plain string. They didn't have to be: any of them can be a JSON object or array instead.

Two reasons to reach for this:

1. **Clarity.** A question with several moving parts reads better as labeled fields than as one run-on sentence.
2. **Sharpening a boundary.** When two options (or two levels, or the yes/no split on a Noul) keep getting confused, adding `what` this option covers, `not_for` what it doesn't, and a couple of `examples` gives the model something concrete to match against — rather than you just writing a longer sentence and hoping.

The field names you choose (`what`, `not_for`, `focus`, `examples` above) aren't part of the API. They're exactly like option names: you pick them, the model reads them.

In [ ]:
import { client } from "../../assets/typesafe-client.ts";
import { choice } from "npm:@typesafe-ai/sdk";

const response = await client.systemOne({
  state:
    "I ordered the standing desk two weeks ago and tracking still says label created. Was I even charged?",
  questions: {
    department: choice(
      {
        question: "Which team should handle this message?",
        focus: "Classify the customer's primary request, not every topic mentioned.",
      },
      {
        billing: {
          what: "Charges, invoices, refunds, or subscriptions",
          not_for: "Order tracking or account access",
          examples: ["I was charged twice", "Where is my refund?"],
        },
        orders: {
          what: "Order status, delivery, cancellation, or returns",
          not_for: "Charges or account access",
          examples: ["Where is my package?", "Cancel my order"],
        },
        account: {
          what: "Login, password, profile, or security",
          not_for: "Charges or delivery",
          examples: ["I can't log in", "Change my email"],
        },
      },
    ),
  },
});

console.log(response.answers.department);

## What just happened

Same `choice()` call you've used since lesson 2 — but both `instructions` and every `criteria` value are objects instead of strings. The ticket genuinely straddles `billing` (was I charged?) and `orders` (where's my package?), which is exactly the kind of boundary the `not_for` fields exist to sharpen.

**Try it yourself:** delete the `not_for` and `examples` fields from all three options, leaving just plain one-line `what` strings, and re-run. Predict whether `confidence` goes up, down, or stays about the same — structure doesn't always help, but it's worth seeing when it does.

## Next up

That's the primitives fully covered: Choice, Score, Noul, and how to structure any of them. Next lesson: Speculative fan-out — composing several questions into one request.

Questions? Ask your agent — it has the same research this lesson was built from, in `research/typesafe-ai.md`.